In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [ ]:
data_a = pd.read_csv("/path/to/clamp_to_homology_times.csv")
data_b = pd.read_csv("/path/to/clamp_to_homology_times.csv")
data_c = pd.read_csv("/path/to/clamp_to_homology_times.csv")
data_d = pd.read_csv("/path/to/clamp_to_homology_times.csv")
# Filter out negative delta cases (this just means that the clamp didn't bind before the homology was found; in contacts simulations, the clamps aren't allowed to bind after homology identification, but no homology is defined for FPT simulations during the run, so the behavior isn't regulated and post-homology-binding clamps are rejected here)
data_a = data_a[data_a["negative_delta_flag"] == 0]
data_b = data_b[data_b["negative_delta_flag"] == 0]
data_c = data_c[data_c["negative_delta_flag"] == 0]
data_d = data_d[data_d["negative_delta_flag"] == 0]

# Time to Clamp Recruit

In [ ]:
time = range(0,1097)
offset = 70 # time between end of contacts delay and enabling of clamp recruitment, used to ensure that start of plot coincides with when clamps can first be recruited

prop_bound_a = []
prop_bound_b = []
prop_bound_c = []
prop_bound_d = []

for t in time:
    prop_bound_a.append(np.sum(data_a["clamp_frame_adjusted"] <= t * 10 + offset)/len(data_a))
    prop_bound_b.append(np.sum(data_b["clamp_frame_adjusted"] <= t * 10 + offset)/len(data_b)) 
    prop_bound_c.append(np.sum(data_c["clamp_frame_adjusted"] <= t * 10 + offset)/len(data_c))
    prop_bound_d.append(np.sum(data_d["clamp_frame_adjusted"] <= t * 10 + offset)/len(data_d))

In [ ]:
def weibull_cdf(x, k, lam):
    return 1 - np.exp(-(x / lam)**k)

t_s = np.array([t * 20/3 for t in time])


datasets = [ # c before b for ordering in plot legend
    (prop_bound_a, "xkcd:dark navy blue", "LEFs Both Chromatids"),
    (prop_bound_c, "xkcd:ochre",          "LEFs Damaged Chromatid"),
    (prop_bound_b, "mediumseagreen",       "LEFs Template Chromatid"),
    (prop_bound_d, "xkcd:ruby",            "No LEFs")
]

x_fit = np.linspace(0, 4000, 1000)

print("=== Fit Parameters (Weibull CDF) ===")
plt.figure(figsize=(8.5, 5.5))

for y_vals, color, label in datasets:
    y_arr = np.array(y_vals)
    
    # Initial guess for optimization: [shape (k), scale (lam)]
    p0 = [0.5, 10000.0]
    
    # Fit the curve
    popt, pcov = curve_fit(weibull_cdf, t_s, y_arr, p0=p0, maxfev=20000)
    k, lam = popt
    perr = np.sqrt(np.diag(pcov))
    
    # Calculate Median (t_50)
    t_50 = lam * np.log(2)**(1/k)
    
    print(f"\n{label}")
    print(f"  k (shape)            = {k:.4f} ± {perr[0]:.4f}")
    print(f"  lam (char. time)     = {lam:.2f} ± {perr[1]:.2f} s")
    print(f"  Median FPT (t_50)    = {t_50:.2f} s")
    
    plt.plot(x_fit, weibull_cdf(x_fit, *popt), color=color, linewidth=2, linestyle='--')
    plt.plot(t_s, y_vals, color=color, alpha=0.6, linewidth=3, label=f"{label} ($t_{{50}}$={t_50:.0f} s)")

plt.rcParams.update({'font.size': 22})
plt.rcParams.update({'axes.titlesize': 24})
plt.xlim(0, 4000)
plt.ylabel("Prop. Clamps Recruited")
plt.xlabel("Time (s)")
plt.title("Clamp Recruitment by LEF Chromatid")
plt.legend(fontsize=19)
plt.show()

# Clamp Recruit to Homology

In [ ]:
time = range(0,7313)

prop_bound_a = []
prop_bound_b = []
prop_bound_c = []
prop_bound_d = []

for t in time:
    prop_bound_a.append(np.sum(data_a["delta_seconds_homology_minus_clamp"] <= t)/len(data_a))
    prop_bound_b.append(np.sum(data_b["delta_seconds_homology_minus_clamp"] <= t)/len(data_b)) 
    prop_bound_c.append(np.sum(data_c["delta_seconds_homology_minus_clamp"] <= t)/len(data_c))
    prop_bound_d.append(np.sum(data_d["delta_seconds_homology_minus_clamp"] <= t)/len(data_d))

In [ ]:
def weibull_cdf(x, k, lam):
    return 1 - np.exp(-(x / lam)**k)

x_data = np.array(time)

datasets = [
    (prop_bound_a, "xkcd:dark navy blue", "Both"),
    (prop_bound_b, "mediumseagreen",       "Template"),
    (prop_bound_c, "xkcd:ochre",          "Damaged"),
    (prop_bound_d, "xkcd:ruby",            "None")
]

x_fit = np.linspace(0, 4000, 1000)

print("=== Fit Parameters (Weibull CDF) ===")
plt.figure(figsize=(8.5, 5.5))

for y_vals, color, label in datasets:
    y_arr = np.array(y_vals)
    
    # Initial guess for optimization: [shape (k), scale (lam)]
    p0 = [0.5, 1000.0]
    
    # Fit the curve
    popt, pcov = curve_fit(weibull_cdf, x_data, y_arr, p0=p0, maxfev=20000)
    k, lam = popt
    perr = np.sqrt(np.diag(pcov))
    
    # Calculate Median (t_50)
    t_50 = lam * np.log(2)**(1/k)
    
    print(f"\n{label}")
    print(f"  k (shape)            = {k:.4f} ± {perr[0]:.4f}")
    print(f"  lam (char. time)     = {lam:.2f} ± {perr[1]:.2f} s")
    print(f"  Median FPT (t_50)    = {t_50:.2f} s")
    
    plt.plot(x_fit, weibull_cdf(x_fit, *popt), color=color, linewidth=2, linestyle='--')
    plt.plot(x_data, y_vals, color=color, alpha=0.6, linewidth=3, label=f"{label} ($t_{{50}}$={t_50:.0f} s)")

plt.rcParams.update({'font.size': 22})
plt.rcParams.update({'axes.titlesize': 24})
plt.ylabel("Prop. Homology Found")
plt.xlabel("Time (s)")
plt.xlim(0,4000)
plt.title("Homology Search After Clamp Recruited")
plt.legend(fontsize=17, borderpad=0.2)
plt.show()

# Recruitment times vs overall search

plt.figure(figsize=(7,5))

plt.rcParams.update({'font.size': 22})
plt.rcParams.update({'axes.titlesize': 24})

plt.hist(data_a["clamp_frame_adjusted"] * 2/3, bins=20, alpha=1, color="xkcd:dark navy blue", density=False, label="Time to Clamp Recruitment")
plt.hist(data_a["homology_frame_adjusted"] * 2/3, bins=100, alpha=0.75, color="xkcd:gray", density=False, label="Time to Homology Find")
plt.legend(fontsize=19)
plt.xlim(0,2000)
plt.xlabel("Time (s)")
plt.ylabel("Count")
plt.title("Clamp and Search Timescales")